In [ ]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [ ]:
with open("data/tinyshakespeare/input.txt", "r", encoding="utf-8") as f:
    text = f.read()

split = int(0.9 * len(text))

train_text = text[:split]
val_text = text[split:]

print(len(train_text), len(val_text))

1003854 111540


In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

print(vocab_size)

65


In [ ]:
from typing import Any


train_data = torch.tensor([stoi[c] for c in train_text], dtype=torch.long)
val_data   = torch.tensor([stoi[c] for c in val_text], dtype=torch.long)

class ShakespeareDataset(Dataset):
    def __init__(self, data, seq_len) -> None:
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, index) -> Any:
        x = self.data[index:index + self.seq_len] # the current letter
        y = self.data[index+ 1: index + self.seq_len + 1] # the next letter

        return x, y


seq_len = 100
batch_size = 64

train_dataset = ShakespeareDataset(train_data, seq_len)
val_dataset = ShakespeareDataset(val_data, seq_len)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
hidden_size = 100
input_size = 65
class GRU(nn.Module):

    def __init__(self):
        super().__init__()
        self.tanh = nn.Tanh()
        self.sigmoid = nn.Sigmoid()
        self.Wx = nn.Parameter(torch.randn(4*hidden_size, input_size))
        self.Wx  = nn.init.xavier_uniform(self.Wx)
        self.Wh = nn.Parameter(torch.randn(4*hidden_size, hidden_size))
        self.Wh  = nn.init.xavier_uniform(self.Wh)
        self.b = nn.Parameter(torch.zeros(4*hidden_size))
        self.Why = nn.Parameter(torch.randn(vocab_size, hidden_size))
        self.By = nn.Parameter(torch.zeros(vocab_size))
        with torch.no_grad():
            self.b[:hidden_size] = 1.0
            
    def forward(self, x, h, c):
        outputs = []

        for t in range(x.shape[1]):
            char = x[:, t]
            char = F.one_hot(char, num_classes=65).float()
            gates = char @ self.Wx.T + h @ self.Wh.T + self.b

            f, i, g, o = gates.chunk(4, dim=1)
            f = self.sigmoid(f)
            i = self.sigmoid(i)
            g = self.tanh(g)
            o = self.sigmoid(o)

            c = f * c + i * g
            h = o * self.tanh(c)
            
            logits = h @ self.Why.T + self.By
            outputs.append(logits)

        
        return torch.stack(outputs, dim=1), h, c


net = GRU()

C:\Users\shehr\AppData\Local\Temp\ipykernel_34128\1184571512.py:12: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  self.Wx  = nn.init.xavier_uniform(self.Wx)
C:\Users\shehr\AppData\Local\Temp\ipykernel_34128\1184571512.py:14: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  self.Wh  = nn.init.xavier_uniform(self.Wh)


In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [ ]:
for epoch in range(15):  # loop over the dataset multiple times

    running_loss = 0.0
    
    for inputs, labels in train_loader:

        H = torch.zeros(inputs.size(0), hidden_size)
        C = torch.zeros(inputs.size(0), hidden_size)
        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs, H, C = net(inputs, H, C) 
        loss = criterion(
            outputs.reshape(-1, 65),
            labels.reshape(-1)
        )
        loss.backward()
        optimizer.step()

        H = H.detach()
        C = C.detach()

        # print statistics
        running_loss += loss.item()
    train_loss = running_loss / len(train_loader)

    net.eval()
    val_loss = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            H = torch.zeros(inputs.size(0), 100)
            C = torch.zeros(inputs.size(0), hidden_size)

            outputs, H, C = net(inputs, H, C) 

            loss = criterion(
                outputs.reshape(-1, vocab_size),
                labels.reshape(-1)
            )

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(
        f"Epoch {epoch+1:2d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    net.train()

print('Finished Training')